In [2]:
! pip install selenium beautifulsoup4 pandas numpy requests lxml openpyxl

In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time

options = Options()
options.add_argument("--start-maximized")

service = Service()
driver = webdriver.Chrome(service=service, options=options)


In [4]:
url = "https://www.nabis.go.kr/termsDetailView.do?menucd=189&gbnCode=S51&eventNo=371"
driver.get(url)
time.sleep(3)  


In [22]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

def build_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--start-maximized")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36")
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(2)
    return driver

def wait_content_growth(driver, selector, timeout=10):
    # 컨테이너 길이가 증가할 때까지 대기 (동적 로딩 대응)
    container = WebDriverWait(driver, timeout).until(EC.presence_of_element_located((By.CSS_SELECTOR, selector)))
    prev_len = len(container.get_attribute("innerHTML") or "")
    for _ in range(timeout * 2):
        curr_len = len(container.get_attribute("innerHTML") or "")
        if curr_len > prev_len:
            return container
        prev_len = curr_len
        driver.implicitly_wait(0.5)
    return container

driver = build_driver()

base_url = "https://k-re100.or.kr/bbs/board.php?bo_table=sub2_2_1&page={}"
all_data = []

for page in range(1, 4):
    driver.get(base_url.format(page))

    # 1) 최상위 컨테이너 대기 + 내용 증가 확인
    root = wait_content_growth(driver, "#fboardlist", timeout=12)

    # 2) 테이블 구조 시도
    table_rows = driver.find_elements(By.CSS_SELECTOR, "#fboardlist div.bo_list_head table tbody tr")
    print(f"[page {page}] table_rows:", len(table_rows))

    # 3) 리스트 구조 대안
    list_items = []
    if len(table_rows) == 0:
        list_items = driver.find_elements(By.CSS_SELECTOR, "#fboardlist div.bo_list ul li")
        print(f"[page {page}] list_items:", len(list_items))

    if len(table_rows) == 0 and len(list_items) == 0:
        # 4) 한 번 더 가시성 대기 (JS 로딩 늦을 때)
        WebDriverWait(driver, 8).until(
            EC.any_of(
                EC.visibility_of_any_elements_located((By.CSS_SELECTOR, "#fboardlist div.bo_list_head table tbody tr")),
                EC.visibility_of_any_elements_located((By.CSS_SELECTOR, "#fboardlist div.bo_list ul li"))
            )
        )
        table_rows = driver.find_elements(By.CSS_SELECTOR, "#fboardlist div.bo_list_head table tbody tr")
        list_items = driver.find_elements(By.CSS_SELECTOR, "#fboardlist div.bo_list ul li")
        print(f"[page {page}] retry rows -> table:{len(table_rows)} list:{len(list_items)}")

    # 5) 추출: 테이블 우선, 없으면 리스트
    if len(table_rows) > 0:
        for tr in table_rows:
            tds = tr.find_elements(By.TAG_NAME, "td")
            if len(tds) < 5:
                continue
            no = tds[0].text.strip()
            # td내부 a가 있을 수도 없을 수도 있음
            company_el = tds[1].find_elements(By.CSS_SELECTOR, "a")
            company = (company_el[0].text if company_el else tds[1].text).strip()
            business_el = tds[2].find_elements(By.CSS_SELECTOR, "a")
            business = (business_el[0].text if business_el else tds[2].text).strip()
            target = tds[3].text.strip()
            join_year = tds[4].text.strip()

            if company:
                all_data.append({
                    "No": no,
                    "기업명": company,
                    "사업영역": business,
                    "RE100 목표/달성률": target,
                    "가입연도": join_year
                })
    elif len(list_items) > 0:
        for li in list_items:
            # li 내부 span 클래스 기반
            no = ""
            try:
                no = li.find_element(By.CSS_SELECTOR, "span.s-number").text.strip()
            except:
                pass
            try:
                company = li.find_element(By.CSS_SELECTOR, "span.s-name a").text.strip()
            except:
                company = li.find_element(By.CSS_SELECTOR, "span.s-name").text.strip()
            try:
                business = li.find_element(By.CSS_SELECTOR, "span.wr_1 a").text.strip()
            except:
                try:
                    business = li.find_element(By.CSS_SELECTOR, "span.wr_1").text.strip()
                except:
                    business = ""
            try:
                target = li.find_element(By.CSS_SELECTOR, "span.wr_2").text.strip()
            except:
                target = ""
            try:
                join_year = li.find_element(By.CSS_SELECTOR, "span.wr_3").text.strip()
            except:
                join_year = ""

            if company:
                all_data.append({
                    "No": no,
                    "기업명": company,
                    "사업영역": business,
                    "RE100 목표/달성률": target,
                    "가입연도": join_year
                })

driver.quit()

df = pd.DataFrame(all_data)
print("총 수집 건수:", len(df))
print(df.head(10))

# 저장
# df.to_csv("01_DataAnalysis/data/policy/k_re100_companies_all.csv", index=False, encoding="utf-8-sig")

[page 1] table_rows: 0
[page 1] list_items: 15
[page 2] table_rows: 0
[page 2] list_items: 15
[page 3] table_rows: 0
[page 3] list_items: 6
총 수집 건수: 36
   No         기업명                             사업영역 RE100 목표/달성률  가입연도
0  36     LS 일렉트릭  전력, 자동화, 스마트에너지(스마트그리드, 마이크로그리…    2040 (0%)  2023
1  35  HD현대사이트솔루션  건설장비, 차량 장비 등(MCV, 선회모터, 주행모터,…    2040 (2%)  2023
2  34       롯데케미칼  기초소재(합성수지, 화성제품, 기초유분제품), 첨단소재…    2050 (4%)  2023
3  33        LG전자  TV, AV, 에어컨, 주방가전, 생활가전, 뷰티, 의…   2050 (10%)  2023
4  32         카카오    카카오톡, 다음, 카카오 T, 멜론 등 모바일 서비스    2040 (3%)  2023
5  31      신한금융그룹     은행, 카드, 증권, 라이프, 손해보험, 리츠 운용   2040 (23%)  2023
6  30       롯데웰푸드       제과, 빙과류, 유제품, 신선식품 제조 및 판매    2040 (0%)  2023
7  29        삼성화재  화재, 해상, 자동차, 배상책임, 장기손해보험, 개인연…    2040 (2%)  2023
8  28        삼성생명               보험, 자산운용, 대출, 헬스케어    2040 (2%)  2023
9  27    삼성바이오로직스                바이오 의약품 위탁생산(CMO)   2050 (25%)  2022


In [ ]:
import os

# 현재 작업 디렉토리 확인
print(os.getcwd())

print(os.listdir("scripts"))

c:\Users\dkreh\Desktop\KDT_RE_5th\3_Project\01_DataAnalysis\scripts\crawling


FileNotFoundError: [WinError 3] 지정된 경로를 찾을 수 없습니다: 'scripts'

In [ ]:
# RE100 해외 사레 분석
import pandas as pd
df = pd.read_csv("../preprocessing/RE100_국가별_비교_확장.csv", encoding="utf-8-sig")
df

,국가,재생에너지 사용률(%)
0,유럽(EU 단일시장),83
1,중국,59
2,미국,60
3,브라질,70
4,인도,39
5,베트남,58
6,일본,36
7,아르헨티나,33
8,인도네시아,33
9,호주,50
